# Hybrid RAG Pipeline

This demo combines BM25, dense similarity, and reranking over a small corpus. Replace the sample documents with PDF chunks to use the same code on your own material.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'llm_lab').exists():
        sys.path.insert(0, str(candidate))
        repo_root = candidate
        break

from llm_lab.demo_data import SAMPLE_DOCS, SAMPLE_QA
from llm_lab.retrieval import hybrid_retrieve

docs = SAMPLE_DOCS + [item['answer'] for item in SAMPLE_QA]
query = 'How does retrieval-augmented generation improve factual answers?'
result = hybrid_retrieve(query, docs, top_k=3)
result

In [ ]:
def pure_vector_retrieve(query, documents, top_k=3):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    vectorizer = TfidfVectorizer().fit(documents + [query])
    matrix = vectorizer.transform(documents + [query])
    scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    order = scores.argsort()[::-1][:top_k]
    return [(documents[index], float(scores[index])) for index in order]

comparison = {
    'hybrid': [(chunk.text, chunk.rerank_score) for chunk in result.results],
    'vector_only': pure_vector_retrieve(query, docs, top_k=3),
}
comparison

In [ ]:
questions = [
    'What technique keeps LoRA training cheap?',
    'Why is a summary buffer useful in a chatbot?',
]
answers = [hybrid_retrieve(question, docs, top_k=2).results[0].text for question in questions]
list(zip(questions, answers))

Swap `SAMPLE_DOCS` for chunked PDF text and keep the rest of the pipeline unchanged to benchmark hybrid retrieval against pure vector search on real factual questions.